In [10]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML

class DualSimplexMaster:
    def __init__(self, c, A, b, name="Задача"):
        self.c = np.array(c, dtype=float)
        self.A = np.array(A, dtype=float)
        self.b = np.array(b, dtype=float)
        self.name = name
        self.num_constraints, self.num_vars = self.A.shape
        
        self.var_names = [f'x{i+1}' for i in range(self.num_vars + self.num_constraints)]
        self.basis = [self.num_vars + i for i in range(self.num_constraints)]
        
        self.full_c = np.concatenate([self.c, np.zeros(self.num_constraints)])
        self.full_A = np.hstack([self.A, np.eye(self.num_constraints)])

    def _smart_round(self, val):
        """Форматирует числа: целые без точки, дроби с 2 знаками."""
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return ""
        if isinstance(val, (int, float, np.float64)):
            if abs(val - round(val)) < 1e-7:
                return str(int(round(val)))
            return f"{val:.2f}"
        return val

    def _render_step(self, pivot_row=None):
        cb = self.full_c[self.basis]
        z_val = np.dot(cb, self.b)
        z_coeffs = np.dot(cb, self.full_A) - self.full_c
        
        data = []
        for i, b_idx in enumerate(self.basis):
            row = [self.var_names[b_idx], cb[i], self.b[i]] + list(self.full_A[i])
            data.append([self._smart_round(x) for x in row])
        
        z_row = ['Zj - Cj', "", z_val] + list(z_coeffs)
        data.append([self._smart_round(x) for x in z_row])
        
        ratio_row = ['(Zj-Cj)/alj', "", ""]
        if pivot_row is not None:
            for j in range(len(self.var_names)):
                val = self.full_A[pivot_row, j]
                if val < -1e-7:
                    ratio_row.append(abs(z_coeffs[j] / val))
                else:
                    ratio_row.append(None)
        else:
            ratio_row.extend([None] * len(self.var_names))
        data.append([self._smart_round(x) for x in ratio_row])

        # Создание уникальной шапки (без объединения ячеек)
        top_row_raw = ["", "", ""] + [self._smart_round(x) for x in self.full_c]
        top_row_unique = [val + (" " * i) for i, val in enumerate(top_row_raw)]
        
        columns = pd.MultiIndex.from_tuples(zip(top_row_unique, ['БП', 'С', 'В'] + self.var_names))
        df = pd.DataFrame(data, columns=columns)

        styles = [
            {'selector': 'th', 'props': [('border', '1px solid black'), ('text-align', 'center'), ('background-color', '#fff')]},
            {'selector': 'td', 'props': [('border', '1px solid gray'), ('text-align', 'center'), ('min-width', '65px')]},
            {'selector': f'tr:nth-child({len(self.basis)+1})', 'props': [('background-color', '#f2f2f2'), ('font-weight', 'bold')]}
        ]
        display(df.style.hide(axis='index').set_table_styles(styles))

    def _print_final_answer(self, status="OPTIMAL"):
        if status == "INFEASIBLE":
            display(HTML("<div style='border: 2px solid red; padding: 10px; background-color: #fff0f0;'><b>Ответ:</b> Решение отсутствует (система ограничений несовместна).</div>"))
            return

        cb = self.full_c[self.basis]
        z_val = np.dot(cb, self.b)
        results = {f'x{i+1}': 0.0 for i in range(len(self.var_names))}
        for i, b_idx in enumerate(self.basis):
            results[self.var_names[b_idx]] = self.b[i]
            
        coords = [self._smart_round(results[f'x{i+1}']) for i in range(self.num_vars)]
        ans_html = f"<b>Ответ:</b><br>X* = ({', '.join(coords)})<br>Z<sub>min</sub> = {self._smart_round(z_val)}"
        display(HTML(f"<div style='border: 2px solid green; padding: 10px; margin-top: 10px; background-color: #f0fff0;'>{ans_html}</div>"))

    def solve(self):
        display(HTML(f"<h2>--- {self.name} ---</h2>"))
        iteration = 0
        while True:
            if np.all(self.b >= -1e-7):
                display(HTML("<b>✅ Оптимальный план найден:</b>"))
                self._render_step()
                self._print_final_answer()
                break
            
            pivot_row = np.argmin(self.b)
            display(HTML(f"<b>Итерация {iteration}</b>"))
            self._render_step(pivot_row)
            
            row_vals = self.full_A[pivot_row]
            ratios = []
            cb = self.full_c[self.basis]
            z_row = np.dot(cb, self.full_A) - self.full_c
            
            for j in range(len(row_vals)):
                if row_vals[j] < -1e-7:
                    ratios.append(abs(z_row[j] / row_vals[j]))
                else:
                    ratios.append(np.inf)
            
            if np.min(ratios) == np.inf:
                self._print_final_answer(status="INFEASIBLE")
                return

            pivot_col = np.argmin(ratios)
            pivot_element = self.full_A[pivot_row, pivot_col]
            self.full_A[pivot_row] /= pivot_element
            self.b[pivot_row] /= pivot_element
            for i in range(self.num_constraints):
                if i != pivot_row:
                    factor = self.full_A[i, pivot_col]
                    self.full_A[i] -= factor * self.full_A[pivot_row]
                    self.b[i] -= factor * self.b[pivot_row]
            
            self.basis[pivot_row] = pivot_col
            iteration += 1

# =================================================================
# ПОЛНЫЙ НАБОР ТЕСТОВЫХ ДАННЫХ
# =================================================================

# 1. СТАНДАРТНОЕ РЕШЕНИЕ (Единственная точка минимума)
DualSimplexMaster(
    c=[1, 1, 2], 
    A=[[1, 1, 1], [-1, 1, 0], [-1, -2, 0]], 
    b=[8, -4, -6],
    name="Тест 1: Решение со скриншота (Optimal)"
).solve()

# 2. НЕСОВМЕСТНОСТЬ (Ограничения противоречат друг другу)
DualSimplexMaster(
    c=[3, 2], 
    A=[[1, 1], [-1, -1]], 
    b=[2, -5],
    name="Тест 2: Отсутствие решения (Infeasible)"
).solve()

# 3. ВЫРОЖДЕННОСТЬ (Несколько прямых пересекаются в одной точке)
DualSimplexMaster(
    c=[10, 20], 
    A=[[-1, -1], [-1, 0], [0, -1]], 
    b=[-5, -5, 0],
    name="Тест 3: Вырожденное решение (Degeneracy)"
).solve()

# 4. АЛЬТЕРНАТИВНЫЙ ОПТИМУМ (Бесконечно много решений)
DualSimplexMaster(
    c=[2, 2], 
    A=[[-1, -1]], 
    b=[-4],
    name="Тест 4: Альтернативный оптимум (Multiple Solutions)"
).solve()

# 5. МИНИМИЗАЦИЯ С ОТРИЦАТЕЛЬНЫМИ КОЭФФИЦИЕНТАМИ
DualSimplexMaster(
    c=[-2, 5], 
    A=[[-1, -1], [1, -2]], 
    b=[-3, -1],
    name="Тест 5: Отрицательные коэффициенты в целевой функции"
).solve()

,,,1,1,2,0,0,0
БП,С,В,x1,x2,x3,x4,x5,x6
x4,0,8,1,1,1,1,0,0
x5,0,-4,-1,1,0,0,1,0
x6,0,-6,-1,-2,0,0,0,1
Zj - Cj,,0,-1,-1,-2,0,0,0
(Zj-Cj)/alj,,,1,0.50,,,,


,,,1,1,2,0,0,0
БП,С,В,x1,x2,x3,x4,x5,x6
x4,0,5,0.50,0,1,1,0,0.50
x5,0,-7,-1.50,0,0,0,1,0.50
x2,1,3,0.50,1,0,0,0,-0.50
Zj - Cj,,3,-0.50,0,-2,0,0,-0.50
(Zj-Cj)/alj,,,0.33,,,,,


,,,1,1,2,0,0,0
БП,С,В,x1,x2,x3,x4,x5,x6
x4,0,2.67,0,0,1,1,0.33,0.67
x1,1,4.67,1,0,0,0,-0.67,-0.33
x2,1,0.67,0,1,0,0,0.33,-0.33
Zj - Cj,,5.33,0,0,-2,0,-0.33,-0.67
(Zj-Cj)/alj,,,,,,,,


,,,3,2,0,0
БП,С,В,x1,x2,x3,x4
x3,0,2,1,1,1,0
x4,0,-5,-1,-1,0,1
Zj - Cj,,0,-3,-2,0,0
(Zj-Cj)/alj,,,3,2,,


,,,3,2,0,0
БП,С,В,x1,x2,x3,x4
x3,0,-3,0,0,1,1
x2,2,5,1,1,0,-1
Zj - Cj,,10,-1,0,0,-2
(Zj-Cj)/alj,,,,,,


...

,,,2,2,0
БП,С,В,x1,x2,x3
x3,0,-4,-1,-1,1
Zj - Cj,,0,-2,-2,0
(Zj-Cj)/alj,,,2,2,


,,,2,2,0
БП,С,В,x1,x2,x3
x1,2,4,1,1,-1
Zj - Cj,,8,0,0,-2
(Zj-Cj)/alj,,,,,


,,,-2,5,0,0
БП,С,В,x1,x2,x3,x4
x3,0,-3,-1,-1,1,0
x4,0,-1,1,-2,0,1
Zj - Cj,,0,2,-5,0,0
(Zj-Cj)/alj,,,2,5,,


,,,-2,5,0,0
БП,С,В,x1,x2,x3,x4
x1,-2,3,1,1,-1,0
x4,0,-4,0,-3,1,1
Zj - Cj,,-6,0,-7,2,0
(Zj-Cj)/alj,,,,2.33,,


,,,-2,5,0,0
БП,С,В,x1,x2,x3,x4
x1,-2,1.67,1,0,-0.67,0.33
x2,5,1.33,0,1,-0.33,-0.33
Zj - Cj,,3.33,0,0,-0.33,-2.33
(Zj-Cj)/alj,,,,,,
